In [53]:
import torch # Torch Module
import torch.nn as nn # Neural Network
import torch.optim as optim # Optimizers
from torch.utils.data import Dataset, DataLoader # For Dataset Loading and Handling

In [54]:
"""
Why "tensor" and not "array" — the actual reason
NumPy arrays don't track gradients or operation history at all — they're purely for numerical computation. 
A PyTorch tensor is functionally a NumPy array that additionally knows how to record its own computational history and 
compute gradients through that history automatically. That gradient-tracking capability is the entire reason PyTorch tensors exist as 
a distinct concept, rather than just using NumPy directly for everything
"""

'\nWhy "tensor" and not "array" — the actual reason\nNumPy arrays don\'t track gradients or operation history at all — they\'re purely for numerical computation. \nA PyTorch tensor is functionally a NumPy array that additionally knows how to record its own computational history and \ncompute gradients through that history automatically. That gradient-tracking capability is the entire reason PyTorch tensors exist as \na distinct concept, rather than just using NumPy directly for everything\n'

In [55]:
# We explicity add a dedcimal because we want the numeric values to be floating numbers and not integers.
# This matters because weights, gradients, and most neural network math need floats, not integers.
x = torch.tensor(
                  [[2., 1., 0., 3.],
                   [1., 0., 2., 1.],
                   [3., 2., 1., 0.],
                   [0., 1., 3., 2.]]
)

In [56]:
""" 
    If shapes don't align for an operation, it throws an error immediately, rather than silently computing something wrong.
    Checking .shape after every operation, especially while learning, is not optional busywork — 
    it's how you verify your mental model of what just happened matches what actually happened.
"""

" \n    If shapes don't align for an operation, it throws an error immediately, rather than silently computing something wrong.\n    Checking .shape after every operation, especially while learning, is not optional busywork — \n    it's how you verify your mental model of what just happened matches what actually happened.\n"

In [57]:
print(x.shape)      # torch.Size([4, 4])
print(x.shape[0])   # 4  — number of rows
print(x.dim())       # 2  — number of dimensions

torch.Size([4, 4])
4
2


In [58]:
# Basic Operations : 

a = torch.tensor([1., 2., 3.])
b = torch.tensor([4., 5., 6.])

print(a + b)        # tensor([5., 7., 9.])   — element-wise addition 
print(a * b)        # tensor([4., 10., 18.]) — element-wise multiplication, NOT dot product ( Hadamard Product )
print(torch.dot(a, b))  # tensor(32.)         — actual dot product: 1×4+2×5+3×6=32

tensor([5., 7., 9.])
tensor([ 4., 10., 18.])
tensor(32.)


In [59]:
# Matrix multiplication —  z = W·x calculation
W = torch.tensor([[0.5, -1.0]])   # shape (1, 2) — our FC layer's weight from the CNN exercise
flat = torch.tensor([[3.0, 1.0]]) # shape (1, 2) — our flattened pooled values

# Matrix multiply: (1,2) can't directly multiply (1,2) — need to transpose
z = flat @ W.T + 0.2
print(z)   # tensor([[0.7000]])

tensor([[0.7000]])


In [60]:
# Autograd - computes partial derivatives for a given tensor automatically if any size.
w = torch.tensor(2.0, requires_grad=True)
x_val = torch.tensor(3.0)

z = w * x_val # The partial derivative : ∂z/∂w = x_val = 3

z.backward()
print(w.grad)   # tensor(3.)

tensor(3.)


In [61]:
# if z = w² + 3w and w = 2, what should dz/dw be at that point (compute the derivative expression first, then plug in w=2)

w = torch.tensor(2.0, requires_grad=True)
z = w**2 + 3*w
z.backward()

print(w.grad)   # tensor(7.)

tensor(7.)


In [62]:
# Building an nn.Module
class SimpleNet(nn.Module):
    def __init__(self):
        super().__init__()
        # nn.Linear(#inputs,#outputs) for that layer
        self.layer1 = nn.Linear(4, 3)   # W¹ is (3,4), matches your hand-derived shape rule
        self.layer2 = nn.Linear(3, 2)   # W² is (2,3)
        self.layer3 = nn.Linear(2, 1)   # W³ is (1,2)
        self.sigmoid = nn.Sigmoid()

    def forward(self, x):
        # Forward Traversing and calculating each layer's output 
        # a0 = X itself
        a1 = self.sigmoid(self.layer1(x))   # a¹ = σ(W¹x + b¹)
        a2 = self.sigmoid(self.layer2(a1))  # a² = σ(W²a¹ + b²)
        y_hat = self.sigmoid(self.layer3(a2))  # ŷ = σ(W³a² + b³)
        return y_hat

In [63]:
model = SimpleNet() # Instantiate the model object

x = torch.tensor([[1.0, 2.0, 3.0, 4.0]])   # shape (1, 4) — batch of 1, 4 features
output = model(x)   # NOT model.forward(x) 
print(output)        # some tensor, shape (1, 1), a random value since weights are randomly initialized

tensor([[0.6272]], grad_fn=<SigmoidBackward0>)


In [64]:
"""
Why model(x) and not model.forward(x) directly — this is a specific PyTorch convention worth locking in now. 
nn.Module overrides Python's __call__ method, and that override does some important bookkeeping (like hooks for training mode, gradient tracking setup) 
around your forward method. Calling model.forward(x) directly skips that bookkeeping. Always call model(x), 
never model.forward(x), even though both will often appear to produce the same output in simple cases
"""

"\nWhy model(x) and not model.forward(x) directly — this is a specific PyTorch convention worth locking in now. \nnn.Module overrides Python's __call__ method, and that override does some important bookkeeping (like hooks for training mode, gradient tracking setup) \naround your forward method. Calling model.forward(x) directly skips that bookkeeping. Always call model(x), \nnever model.forward(x), even though both will often appear to produce the same output in simple cases\n"

In [65]:
# Let's inspect the model parameters
for name, param in model.named_parameters():
    print(f'name: {name}\t param_shape: {param.shape}')

name: layer1.weight	 param_shape: torch.Size([3, 4])
name: layer1.bias	 param_shape: torch.Size([3])
name: layer2.weight	 param_shape: torch.Size([2, 3])
name: layer2.bias	 param_shape: torch.Size([2])
name: layer3.weight	 param_shape: torch.Size([1, 2])
name: layer3.bias	 param_shape: torch.Size([1])


In [66]:
# This we did was jut a forward pass only using forward method.

In [67]:
# Another example of this

In [68]:
class SimpleNet (nn.Module):
    def __init__(self):
        super().__init__()
        # Creating the neural layers
        self.layer1 = nn.Linear(4,3)
        self.layer2 = nn.Linear(3,2)
        self.layer3 = nn.Linear(2,1)
        self.sigmoid = nn.Sigmoid()

    def forward(self,X):
        # Forward Traversal and their outputs
        a1 = self.sigmoid(self.layer1(X))
        a2 = self.sigmoid(self.layer2(a1))
        y_hat = self.sigmoid(self.layer3(a2))
        
        return y_hat
    
model = SimpleNet()
X = torch.tensor([[1.0, 2.0, 3.0, 4.0]])

print(f' y_hat : {model(X)} \n with shape : {model(X).shape} ')

# OP : 
# y_hat : tensor([[0.4233]], grad_fn=<SigmoidBackward0>) 
# with shape : torch.Size([1, 1]) 

 y_hat : tensor([[0.5344]], grad_fn=<SigmoidBackward0>) 
 with shape : torch.Size([1, 1]) 


In [69]:
# Setting up the criterion
criterion = nn.MSELoss()

# Setting up the optimizer
optimizer = optim.Adam(model.parameters(),lr=0.01)

# we pass parameters to update them on their history of values.

In [70]:
# Training Loop
x_train = torch.tensor([[1.0, 2.0, 3.0, 4.0]])
y_train = torch.tensor([[1.0]])   # pretend the "correct" answer is 1.0

# Running this cell again and again causes model to retrain #clicks
for epoch in range(1,101):
    optimizer.zero_grad()          # step 1
    y_pred = model(x_train)        # step 2 — forward pass
    loss = criterion(y_pred, y_train)   # step 3 — compute loss
    loss.backward()                # step 4 — backward pass
    optimizer.step()               # step 5 — update weights

    if epoch % 20 == 0:
        print(f"Epoch {epoch}, Loss: {loss.item():.4f}")

# Note :  Calling optimizer.zero_grad() resets every parameter's .grad back to zero right before each new backward pass, 
#         ensuring each epoch's gradient calculation is clean and isolated 



Epoch 20, Loss: 0.1411
Epoch 40, Loss: 0.0885
Epoch 60, Loss: 0.0541
Epoch 80, Loss: 0.0340
Epoch 100, Loss: 0.0228


In [71]:
# Building a neural network on tiny dataset

In [72]:
class ToyDataset(Dataset):
    def __init__(self, n_samples=100):
        # Random 4-feature inputs, and a synthetic target: 1 if sum > 8, else 0
        self.X = torch.rand(n_samples, 4) * 4        # random values roughly in [0,4)
        self.y = (self.X.sum(dim=1) > 8).float().unsqueeze(1)   # shape (n_samples, 1)

    def __len__(self):
        return len(self.X)

    def __getitem__(self, idx):
        return self.X[idx], self.y[idx]

dataset = ToyDataset(n_samples=200)
train_loader = DataLoader(dataset, batch_size=16, shuffle=True)

In [73]:
model = SimpleNet()
criterion = nn.MSELoss()
optimizer = optim.Adam(model.parameters(), lr=0.01)

for epoch in range(51):
    epoch_loss = 0.0
    for batch_X, batch_y in train_loader:
        optimizer.zero_grad()
        y_pred = model(batch_X)
        loss = criterion(y_pred, batch_y)
        loss.backward()
        optimizer.step()
        epoch_loss += loss.item()

    if epoch % 10 == 0:
        avg_loss = epoch_loss / len(train_loader)
        print(f"Epoch {epoch}, Avg Loss: {avg_loss:.4f}")

Epoch 0, Avg Loss: 0.2494
Epoch 10, Avg Loss: 0.2359
Epoch 20, Avg Loss: 0.1911
Epoch 30, Avg Loss: 0.1177
Epoch 40, Avg Loss: 0.0748
Epoch 50, Avg Loss: 0.0558
